In [1]:
# Install required dependencies
!pip install -U -q "transformers==4.35.2" "bitsandbytes==0.42.0"
!pip install -q -U faiss-cpu tiktoken sentence-transformers langchain-community langchain-huggingface
!pip install -q biopython
!pip install -U accelerate>=0.26.0

# Import necessary libraries
import os
import random
import torch
from transformers import AutoTokenizer, BertForSequenceClassification, pipeline
from Bio import SeqIO
from langchain.schema import Document
from langchain.embeddings import CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline

####################################
# 1. Create and Process DNA Data with Metadata
####################################
def calculate_gc_content(sequence):
    gc_count = sequence.count('G') + sequence.count('C')
    return (gc_count / len(sequence)) * 100

def generate_dna_sequence(length):
    return ''.join(random.choice('ATCG') for _ in range(length))

# Generate 100 sequences with variable lengths between 800 and 1200
dna_data = []
for i in range(100):
    seq_length = random.randint(800, 1200)
    seq = generate_dna_sequence(seq_length)
    gc = calculate_gc_content(seq)
    dna_data.append((f"seq_{i}", seq, seq_length, gc))

# Save sequences to a FASTA file
fasta_file = "dna_sequences.fasta"
with open(fasta_file, "w") as f:
    for seq_id, seq, length, gc in dna_data:
        f.write(f">{seq_id}\n{seq}\n")

# Load sequences using BioPython
records = list(SeqIO.parse(fasta_file, "fasta"))

# Convert sequences into LangChain Documents with structured metadata
dna_documents = []
for record in records:
    seq_str = str(record.seq)
    length = len(seq_str)
    gc = calculate_gc_content(seq_str)
    # Create a document with detailed information
    doc_text = (f"Sequence ID: {record.id}\n"
                f"Sequence: {seq_str}\n"
                f"Length: {length}\n"
                f"GC Content: {gc:.2f}%")
    doc = Document(page_content=doc_text, metadata={"id": record.id, "length": length, "gc": gc})
    dna_documents.append(doc)

print(f"Number of DNA documents: {len(dna_documents)}")

####################################
# 2. Set Up FAISS Vector Store for Retrieval
####################################
store = LocalFileStore("./cache/")
embed_model_id = 'sentence-transformers/all-MiniLM-L6-v2'
core_embeddings_model = HuggingFaceEmbeddings(model_name=embed_model_id)
embedder = CacheBackedEmbeddings.from_bytes_store(core_embeddings_model, store, namespace=embed_model_id)
vector_store = FAISS.from_documents(dna_documents, embedder)

# Test the vector store with a query
test_query = "Find sequences with high GC content"
embedding_vector = core_embeddings_model.embed_query(test_query)
docs = vector_store.similarity_search_by_vector(embedding_vector, k=4)
print("Retrieved Documents (sample):")
for page in docs:
    print(page.page_content)
    print("-" * 50)

####################################
# 3. (Optional) Load DNABERT for Domain-Specific Embedding
####################################
# We load DNABERT (using BertForSequenceClassification) to demonstrate how to get a domain‐specific embedding.
# (This embedding isn’t used by the QA chain below, but you could incorporate it in custom downstream tasks.)
tokenizer_dnabert = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
model_dnabert = BertForSequenceClassification.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

def dna_sequence_to_embedding(sequence):
    encoded_input = tokenizer_dnabert(sequence, return_tensors="pt")
    with torch.no_grad():
        output = model_dnabert(**encoded_input)
    # Using logits as a simplistic embedding representation
    return output.logits.squeeze().numpy()

# Test DNABERT embedding on a sample sequence
sample_sequence = "ATGCGTACGAGT"
sample_embedding = dna_sequence_to_embedding(sample_sequence)
print("Sample DNABERT embedding:", sample_embedding)

####################################
# 4. Use a Local Generative Model for Final Question Answering
####################################
# Instead of using an external API, we use a local text-to-text generation pipeline.
hf_pipeline = pipeline("text2text-generation", model="google/flan-t5-large")
# Wrap the pipeline in LangChain's HuggingFacePipeline wrapper.
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Create a RetrievalQA chain using the vector store as retriever.
# Using chain_type="map_reduce" to better combine info from multiple documents.
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="map_reduce",
    retriever=vector_store.as_retriever()
)

####################################
# 5. Improved Bioinformatics Queries
####################################
bio_queries = [
    "What is the average GC content of the DNA sequences in the dataset?",
    "How many sequences have a GC content greater than 50%?",
    "Which sequence has the highest GC content?",
    "What is the distribution of sequence lengths in the dataset?",
    "Identify any repetitive motifs in these DNA sequences."
]

print("Final QA Chain Outputs:")
for query in bio_queries:
    print(f"Question: {query}")
    answer = qa_chain.run(query)
    print(f"Answer: {answer}")
    print("-" * 50)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 32.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.0 MB/s eta 0:00:

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1497 > 1024). Running this sequence through the model will result in indexing errors


Answer: 52.28%
--------------------------------------------------
Question: How many sequences have a GC content greater than 50%?
Answer: GC Content: 49.22%
--------------------------------------------------
Question: Which sequence has the highest GC content?
Answer: GGCCCACGTAAGGTATATCAGCACGT
--------------------------------------------------
Question: What is the distribution of sequence lengths in the dataset?
Answer: Length: 870 GC Content: 48.97% Length: 1186
--------------------------------------------------
Question: Identify any repetitive motifs in these DNA sequences.
Answer: GCGCGGGGCACGGTAAGTCTGGCCATGGTTACAGGGATAGT
--------------------------------------------------
